Sau khi kết nối thành công ở Bước 2, bạn có thể bắt tay ngay vào học và thực hành Buổi 5:

Trong Jupyter Lab, mở file 02_read_bronze.ipynb (file hiện tại đang trống).
Viết đoạn code khởi tạo SparkSession (tương tự như file 01_connect_spark.ipynb).
Thực hành đọc dữ liệu từ Bronze layer và thực hiện các thao tác biến đổi DataFrame cơ bản:
Schema & Cast: Ép kiểu các cột ngày tháng (ví dụ: order_purchase_timestamp) từ String sang Timestamp.
Filter & Select: Lọc các đơn hàng theo trạng thái (order_status), chọn các cột cần thiết.
Duplicates: Sử dụng dropDuplicates() để làm sạch các dòng dữ liệu trùng lặp.
Explain: Thêm .explain() vào cuối DataFrame để đọc Execution Plan (Kế hoạch thực thi) của Spark (đây là nội dung chính của buổi 5).

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (SparkSession.builder
    .appName("Dataframe practice")
    .master("local[*]")
    .getOrCreate()
)

print("Session started succesfully")

Session started succesfully


In [3]:
import os
csv_path = os.path.join("..", "data", "raw", "olist_orders_dataset.csv")

df_raw = spark.read.csv(csv_path, header=True, inferSchema=True)
df_raw.printSchema()

df_raw.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)

+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------

In [5]:
df_selected = df_raw.select(
    col("order_id"),
    col("customer_id"),
    col("order_purchase_timestamp").alias("purchase_timestamp"),
    col("order_status").alias("status")
)

df_selected.show(5, truncate=False)

+--------------------------------+--------------------------------+-------------------+---------+
|order_id                        |customer_id                     |purchase_timestamp |status   |
+--------------------------------+--------------------------------+-------------------+---------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|2017-10-02 10:56:33|delivered|
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|2018-07-24 20:41:37|delivered|
|47770eb9100c2d0c44946d9cf07ec65d|41ce2a54c0b03bf3443c3d931a367089|2018-08-08 08:38:49|delivered|
|949d5b44dbf5de918fe9c16f97b45f8a|f88197465ea7920adcdbec7375364d82|2017-11-18 19:28:06|delivered|
|ad21c59c0840e6cb83a9ceb5573f8159|8ab97904e6daea8866dbdbc4fb7aad2c|2018-02-13 21:18:39|delivered|
+--------------------------------+--------------------------------+-------------------+---------+
only showing top 5 rows


In [7]:
df_casted = df_selected \
    .withColumnRenamed("purchase_timestamp", "timestamp") \
    .withColumn("timestamp", col("timestamp").cast("timestamp"))

df_casted.printSchema()
df_casted.show(5, truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- status: string (nullable = true)

+--------------------------------+--------------------------------+-------------------+---------+
|order_id                        |customer_id                     |timestamp          |status   |
+--------------------------------+--------------------------------+-------------------+---------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|2017-10-02 10:56:33|delivered|
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|2018-07-24 20:41:37|delivered|
|47770eb9100c2d0c44946d9cf07ec65d|41ce2a54c0b03bf3443c3d931a367089|2018-08-08 08:38:49|delivered|
|949d5b44dbf5de918fe9c16f97b45f8a|f88197465ea7920adcdbec7375364d82|2017-11-18 19:28:06|delivered|
|ad21c59c0840e6cb83a9ceb5573f8159|8ab97904e6daea8866dbdbc4fb7aad2c|2018-02-13 21:18:39|delivered|
+--------------------------------+-----------

In [8]:
df_filtered = df_casted.filter(col("status") == "delivered")

df_filtered.show(5, truncate=False)
print(f"Total order delivered: {df_filtered.count()}")

+--------------------------------+--------------------------------+-------------------+---------+
|order_id                        |customer_id                     |timestamp          |status   |
+--------------------------------+--------------------------------+-------------------+---------+
|e481f51cbdc54678b7cc49136f2d6af7|9ef432eb6251297304e76186b10a928d|2017-10-02 10:56:33|delivered|
|53cdb2fc8bc7dce0b6741e2150273451|b0830fb4747a6c6d20dea0b8c802d7ef|2018-07-24 20:41:37|delivered|
|47770eb9100c2d0c44946d9cf07ec65d|41ce2a54c0b03bf3443c3d931a367089|2018-08-08 08:38:49|delivered|
|949d5b44dbf5de918fe9c16f97b45f8a|f88197465ea7920adcdbec7375364d82|2017-11-18 19:28:06|delivered|
|ad21c59c0840e6cb83a9ceb5573f8159|8ab97904e6daea8866dbdbc4fb7aad2c|2018-02-13 21:18:39|delivered|
+--------------------------------+--------------------------------+-------------------+---------+
only showing top 5 rows
Total order delivered: 96478


In [9]:
df_filtered.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('status, delivered)
+- Project [order_id#17, customer_id#18, cast(timestamp#80 as timestamp) AS timestamp#81, status#60]
   +- Project [order_id#17, customer_id#18, purchase_timestamp#59 AS timestamp#80, status#60]
      +- Project [order_id#17, customer_id#18, order_purchase_timestamp#20 AS purchase_timestamp#59, order_status#19 AS status#60]
         +- Relation [order_id#17,customer_id#18,order_status#19,order_purchase_timestamp#20,order_approved_at#21,order_delivered_carrier_date#22,order_delivered_customer_date#23,order_estimated_delivery_date#24] csv

== Analyzed Logical Plan ==
order_id: string, customer_id: string, timestamp: timestamp, status: string
Filter (status#60 = delivered)
+- Project [order_id#17, customer_id#18, cast(timestamp#80 as timestamp) AS timestamp#81, status#60]
   +- Project [order_id#17, customer_id#18, purchase_timestamp#59 AS timestamp#80, status#60]
      +- Project [order_id#17, customer_id#18, order_purchase_times